---
## OpenShift DB — Direct Query

Uses `oc port-forward` to tunnel to the PostgreSQL services inside the cluster, then queries via psycopg2.

**Prerequisites**
```
oc login ...   # must be logged in
```

| Key | Value |
|-----|-------|
| Namespace | `rit-genai-naga-dev` |
| Pipeline DB | `rit-genai-dev-mastering-homework` via `rit-genai-naga-dev-primary` |
| OpenWebUI DB | `pilotgenai_dev_pg` via `rit-genai-naga-dev-pgbouncer` |

### Via `oc exec` into the app pod (no port-forward needed)

The deployment pod already has psycopg2 + `PIPELINE_DATABASE_URL` set, so you can run queries directly inside it.

In [2]:
import subprocess, json, io
import pandas as pd

NAMESPACE      = "rit-genai-naga-dev"
APP_DEPLOYMENT = "open-webui-mastering-homework"

def _get_pod():
    out = subprocess.check_output(
        ["oc", "get", "pods", "-n", NAMESPACE,
         "-l", f"app={APP_DEPLOYMENT}",
         "--field-selector=status.phase=Running",
         "-o", "jsonpath={.items[0].metadata.name}"],
        text=True,
    ).strip()
    if not out:
        raise RuntimeError(f"No running pod found for deployment '{APP_DEPLOYMENT}'")
    return out

def exec_sql(sql, db="pipeline", pod=None):
    """Run SQL inside the app pod via python3 — no port-forward needed.
    db='pipeline'  -> PIPELINE_DATABASE_URL  (rit-genai-dev-mastering-homework)
    db='openwebui' -> DATABASE_URL           (pilotgenai_dev_pg)
    Returns a DataFrame.
    """
    if pod is None:
        pod = _get_pod()
        print(f"Pod: {pod}")

    env_var = "PIPELINE_DATABASE_URL" if db == "pipeline" else "DATABASE_URL"

    py_script = (
        "import os, json, psycopg2, psycopg2.extras\n"
        f"conn = psycopg2.connect(os.environ[{repr(env_var)}])\n"
        "cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)\n"
        f"cur.execute({repr(sql)})\n"
        "if cur.description:\n"
        "    rows = [dict(r) for r in cur.fetchall()]\n"
        "    print(json.dumps(rows, default=str))\n"
        "else:\n"
        "    conn.commit()\n"
        "    print(json.dumps({'rowcount': cur.rowcount}))\n"
        "conn.close()\n"
    )

    result = subprocess.run(
        ["oc", "exec", pod, "-n", NAMESPACE, "--", "python3", "-c", py_script],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError("Query failed inside pod")

    data = json.loads(result.stdout)
    if isinstance(data, list):
        df = pd.DataFrame(data)
        display(df)
        print(f"\n{len(df)} row(s)")
        return df
    else:
        print(f"OK — rows affected: {data['rowcount']}")
        return data



# Table Config

In [13]:
exec_sql("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' ORDER BY table_name;");

Pod: open-webui-mastering-homework-7c8bf8d574-sv4zr


,table_name
0,general_prompt
1,pipeline_job
2,student_analysis
3,student_conversation
4,student_practice_assignment
5,student_question_evaluations
6,student_topic_performance
7,tutor_error_type
8,tutor_homework
9,tutor_practice_problem



11 row(s)


In [30]:
exec_sql("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' ORDER BY table_name;", db="openwebui");

Pod: open-webui-mastering-homework-7c8bf8d574-sv4zr


,table_name
0,alembic_version
1,auth
2,channel
3,channel_member
4,chat
5,chatidtag
6,config
7,document
8,document_chunk
9,feedback



24 row(s)


# HW db

In [ ]:
exec_sql("""SELECT * FROM tutor_homework""");

Pod: open-webui-mastering-homework-7c8bf8d574-sv4zr


,id,group_id,model_id,question_data,answer_data,answer_source,topic_mapping,question_uploaded_at,answer_uploaded_at,topic_mapped_at
0,53d1a54e-2b16-4ea7-b101-53f2281db012,1cf1b19f-f173-4f64-a050-338a54924325,math1011-spring-2026-hw-9-model,```markdown\n# MATH-UA 121: Calculus I \n**Fa...,None,None,"{'1': ['Limits at Infinity'], '2': ['Limits'],...",2026-03-30 05:31:33.566858,None,2026-03-30 05:31:36.358809



1 row(s)


In [3]:
# get student analysis
exec_sql("""SELECT * FROM student_practice_assignment""");

Pod: open-webui-mastering-homework-6bb86b87f4-xbbrj


,id,student_id,student_email,homework_id,practice_problem_id,assigned_items,created_at
0,d6bdba6b-d677-441b-92ba-719d616f17b4,c9efa19c-c395-468d-9e3e-45b04865243c,mathstudent2@gmail.com,fc23d5e4-4f6f-4f32-b95c-eee747063335,2c656018-3307-4ff0-9381-726c6f77ee59,"[{'number': 1, 'text': '**1.** Evaluate the fo...",2026-03-30 06:53:11.250961
1,5cb6a499-cb60-4649-b189-95a4c3744b72,cd6cfa15-7960-47b6-a802-7daf8bea8609,mathstudent1@gmail.com,fc23d5e4-4f6f-4f32-b95c-eee747063335,2c656018-3307-4ff0-9381-726c6f77ee59,"[{'number': 1, 'text': '**1.** Evaluate the fo...",2026-03-30 06:53:11.251028



2 row(s)


# Openwebui db

## Group Related

In [ ]:
# Get all custom models assigned to the group
group_name = '%Math1011 Spring 2026%'

exec_sql(f"""
SELECT
  g.id        AS group_id,
  g.user_id   AS creator_id,
  u.name      AS creator_name,
  g.name      AS group_name,
  m.id        AS model_id,
  m.base_model_id,
  m.is_active
FROM model m
JOIN "group" g
  ON (m.access_control->'read'->'group_ids')::jsonb ? g.id
JOIN "user" u
  ON g.user_id = u.id
WHERE g.name LIKE '{group_name}'
ORDER BY g.name, m.id;
""", db="openwebui");


Pod: open-webui-mastering-homework-7c8bf8d574-sv4zr


,group_id,creator_id,creator_name,group_name,model_id,base_model_id,is_active
0,1cf1b19f-f173-4f64-a050-338a54924325,39c69148-99a0-468b-b9ba-794cb555b509,Math Prof,Math1011 Spring 2026,math1011-spring-2026-hw-7-model,llm_0327_mathprof1.@gpt-4o/gpt-4o,True
1,1cf1b19f-f173-4f64-a050-338a54924325,39c69148-99a0-468b-b9ba-794cb555b509,Math Prof,Math1011 Spring 2026,-math1011-spring-2026-hw-9-model,llm_0327_mathprof1.@gpt-4o/gpt-4o,True



2 row(s)


# Other

In [7]:
# Count the number of chats per group:
group_name = '%Math1011 Spring 2026%'

exec_sql(f"""
SELECT g.id, g.name, COUNT(*) FROM "chat" c JOIN "group" g ON c.group_id = g.id WHERE g.name LIKE '{group_name}' GROUP BY g.id, g.name; 
""", db="openwebui");


Pod: open-webui-mastering-homework-7c8bf8d574-sv4zr


,id,name,count
0,1cf1b19f-f173-4f64-a050-338a54924325,Math1011 Spring 2026,17



1 row(s)


In [ ]:
# Get the role for the user
exec_sql("""SELECT id, name, email FROM "user" WHERE email LIKE '%math%' LIMIT 20""", db="openwebui");

Pod: open-webui-mastering-homework-7c8bf8d574-sv4zr


,id,name,email
0,39c69148-99a0-468b-b9ba-794cb555b509,Math Prof,mathprof1@gmail.com
1,cd6cfa15-7960-47b6-a802-7daf8bea8609,Peter Smith,mathstudent1@gmail.com
2,c9efa19c-c395-468d-9e3e-45b04865243c,Mary Jones,mathstudent2@gmail.com



3 row(s)
